# Section 5.2 Perturbation response prediction with sci-Plex

This notebook isolates the Chapter 5 perturbation-response application. It evaluates condition-aware sci-Plex A549 response prediction under the held-out dose and held-out compound splits only. The single-cell time-course EB experiments are handled by `05_1_single_cell_timecourse_main_suite.ipynb`.


## Setup


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_ch05_perturbation")

from pathlib import Path
import sys
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
except Exception as exc:
    raise ImportError("Chapter 5 perturbation experiments require PyTorch.") from exc

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/home/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    load_or_prepare_sciplex3_a549,
    load_lincs_smiles_corpus,
    make_sciplex_split,
    make_sciplex_pca_state_table,
    compute_rdkit2d_with_external_norm,
)
from src.ch05_experiments import (
    set_global_seed,
    choose_heldout_compound,
    sciplex_split_counts,
    evaluate_sciplex_split,
    aggregate_metric_table,
)

DEFAULT_SEED = int(os.environ.get("CH05_SEED", "42"))
QUICK_MODE = os.environ.get("CH05_QUICK", "0") == "1"
TRAINING_STEPS = int(os.environ.get("CH05_TRAINING_STEPS", "6000"))
BATCH_SIZE = int(os.environ.get("CH05_BATCH_SIZE", "256"))
NFE = int(os.environ.get("CH05_NFE", "32"))
SCIPLEX_DOWNLOAD_IN_CH05 = os.environ.get("CH05_SCIPLEX_DOWNLOAD_IN_CH05", "0") == "1"
SCIPLEX_SYNTHETIC_IF_MISSING = os.environ.get("CH05_ALLOW_SYNTHETIC_SCIPLEX", "0") == "1"
MAX_EVAL_GROUPS = os.environ.get("CH05_MAX_EVAL_GROUPS", "")
MAX_EVAL_GROUPS = None if MAX_EVAL_GROUPS == "" else int(MAX_EVAL_GROUPS)

set_global_seed(DEFAULT_SEED)
random.seed(DEFAULT_SEED)
np.random.seed(DEFAULT_SEED)
torch.manual_seed(DEFAULT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(DEFAULT_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "figures" / "ch05"
TABLE_DIR = PROJECT_ROOT / "tables" / "ch05"
OUT_DIR = PROJECT_ROOT / "outputs" / "ch05"
for path in [FIG_DIR, TABLE_DIR, OUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def json_ready(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    return obj

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True))
    return path

def save_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path

def save_figure(fig, filename):
    path = FIG_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

RUN_SUMMARY = {
    "experiment": "sci-Plex perturbation response prediction",
    "scope": "Split B held-out highest dose and Split C held-out compound only",
    "quick_mode": bool(QUICK_MODE),
    "seed": int(DEFAULT_SEED),
    "device": str(DEVICE),
    "training_steps": int(TRAINING_STEPS),
    "batch_size": int(BATCH_SIZE),
    "nfe": int(NFE),
    "paths": {"figures": str(FIG_DIR), "tables": str(TABLE_DIR), "outputs": str(OUT_DIR)},
}
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}; quick={QUICK_MODE}; steps={TRAINING_STEPS}; batch={BATCH_SIZE}; nfe={NFE}")
print("Evaluated splits: Split B held-out highest dose; Split C held-out compound")


## 1. sci-Plex data audit and split-aware preprocessing


In [ ]:
try:
    sciplex = load_or_prepare_sciplex3_a549(
        data_dir=DATA_DIR / "sciplex3_a549",
        lincs_smiles_dir=DATA_DIR / "chemcpa_lincs_smiles",
        download=SCIPLEX_DOWNLOAD_IN_CH05,
        synthetic_if_missing=SCIPLEX_SYNTHETIC_IF_MISSING,
        hvg_top_n=1000,
        seed=DEFAULT_SEED,
    )
except FileNotFoundError as exc:
    raise FileNotFoundError(
        "sci-Plex cache is missing. Run the final Chapter 4 cache section first, "
        "or set CH05_SCIPLEX_DOWNLOAD_IN_CH05=1 for this notebook."
    ) from exc

metadata = sciplex.metadata.reset_index(drop=True).copy()
source_text = str(sciplex.summary.get("source", ""))
if bool(sciplex.summary.get("is_synthetic", False)) or "synthetic" in source_text.lower():
    save_json(OUT_DIR / "real_data_audit.json", {
        "status": "failed",
        "reason": "sci-Plex cache is synthetic or synthetic-labeled",
        "summary": sciplex.summary,
    })
    raise ValueError("Chapter 5 full run refuses synthetic sci-Plex data. Rebuild the Chapter 4 cache section with real A549 data.")
heldout_compound, heldout_reason = choose_heldout_compound(metadata)
split_b = make_sciplex_split("heldout_highest_dose", metadata, seed=DEFAULT_SEED)
split_c = make_sciplex_split("heldout_compound", metadata, heldout_compound=heldout_compound, seed=DEFAULT_SEED)
splits = {
    "Split B held-out highest dose": split_b,
    "Split C held-out compound": split_c,
}
split_table = sciplex_split_counts(splits)
save_csv(TABLE_DIR / "tab_5_2_sciplex_splits.csv", split_table)

states = {}
for split_name, split_meta in splits.items():
    states[split_name] = make_sciplex_pca_state_table(sciplex.adata, split_meta, n_pcs=30, hvg_top_n=1000)

cell_counts = sciplex.cell_counts.copy()
vehicle_count = int(metadata["is_vehicle"].sum())
compound_count = int(metadata.loc[~metadata["is_vehicle"], "compound"].nunique())
RUN_SUMMARY["sciplex_data"] = {
    "paths": sciplex.paths,
    "summary": sciplex.summary,
    "K_compounds": compound_count,
    "vehicle_count": vehicle_count,
    "compound_dose_counts_head": cell_counts.head(40),
    "heldout_compound": heldout_compound,
    "heldout_compound_reason": heldout_reason,
    "split_counts": split_table,
    "pca_explained_variance": {name: state.pca_explained_variance_ratio for name, state in states.items()},
}
real_data_audit = {
    "status": "ok",
    "source": sciplex.summary.get("source"),
    "source_url": sciplex.summary.get("source_url"),
    "is_synthetic": bool(sciplex.summary.get("is_synthetic", False)),
    "K_compounds": compound_count,
    "compound_list": sciplex.summary.get("compound_list", sorted(metadata.loc[~metadata["is_vehicle"], "compound"].astype(str).unique().tolist())),
    "vehicle_count": vehicle_count,
    "dose_values": sciplex.summary.get("dose_values", sorted(map(float, metadata.loc[~metadata["is_vehicle"], "dose"].dropna().unique().tolist()))),
    "missing_smiles_count": sciplex.summary.get("missing_smiles_count"),
    "obs_schema_used": sciplex.summary.get("obs_schema_used"),
    "subset_rule": sciplex.summary.get("subset_rule"),
    "split_counts": split_table,
}
save_json(OUT_DIR / "real_data_audit.json", real_data_audit)
print("K compounds:", compound_count, "vehicle cells:", vehicle_count)
print("Held-out compound:", heldout_compound, heldout_reason)
display(cell_counts.head(30))
display(split_table)


## 2. RDKit2D preprocessing audit


In [ ]:
lincs = load_lincs_smiles_corpus(cache_dir=DATA_DIR / "chemcpa_lincs_smiles", download=SCIPLEX_DOWNLOAD_IN_CH05)
compound_smiles = (
    metadata.loc[~metadata["is_vehicle"], ["compound", "SMILES"]]
    .drop_duplicates()
    .sort_values("compound")
    .reset_index(drop=True)
)
rdkit_cache_path = OUT_DIR / "rdkit2d_compound_features.npz"
rdkit_diag_path = OUT_DIR / "rdkit2d_diagnostics.json"
if rdkit_cache_path.exists() and rdkit_diag_path.exists():
    z = np.load(rdkit_cache_path, allow_pickle=True)
    cached_compounds = z["compounds"].astype(str).tolist()
    current_compounds = compound_smiles["compound"].astype(str).tolist()
    if cached_compounds == current_compounds:
        rdkit_features = z["features"].astype(np.float32)
        rdkit_diagnostics = json.loads(rdkit_diag_path.read_text())
        if int(rdkit_diagnostics.get("D_RDKit", rdkit_features.shape[1])) != int(rdkit_features.shape[1]):
            rdkit_features = None
            rdkit_diagnostics = None
    else:
        rdkit_cache_path.unlink()
        rdkit_diag_path.unlink()
        rdkit_features = None
        rdkit_diagnostics = None
else:
    rdkit_features = None
    rdkit_diagnostics = None
if rdkit_features is None:
    rdkit_result = compute_rdkit2d_with_external_norm(
        compound_smiles["SMILES"].tolist(),
        external_smiles=lincs.smiles,
    )
    rdkit_features = rdkit_result.features
    rdkit_diagnostics = rdkit_result.diagnostics
    rdkit_diagnostics["D_RDKit"] = int(rdkit_features.shape[1])
    np.savez_compressed(
        rdkit_cache_path,
        compounds=compound_smiles["compound"].astype(str).to_numpy(),
        features=rdkit_features,
    )
    save_json(rdkit_diag_path, rdkit_diagnostics)
rdkit_by_compound = {
    str(compound): rdkit_features[i]
    for i, compound in enumerate(compound_smiles["compound"].astype(str).tolist())
}
rdkit_audit = pd.DataFrame([rdkit_diagnostics])
save_csv(OUT_DIR / "rdkit2d_audit.csv", rdkit_audit)
RUN_SUMMARY["rdkit2d"] = {
    **rdkit_diagnostics,
    "D_RDKit": int(rdkit_features.shape[1]),
    "lincs_smiles_path": str(lincs.path),
    "lincs_smiles_count": len(lincs.smiles),
    "lincs_invalid_count": int(lincs.n_invalid),
}
display(rdkit_audit)


## 3. Split B held-out highest dose


In [ ]:
split_b_metrics, split_b_cache = evaluate_sciplex_split(
    states["Split B held-out highest dose"].X_pca,
    states["Split B held-out highest dose"].metadata,
    rdkit_by_compound=rdkit_by_compound,
    split_name="Split B held-out highest dose",
    training_steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE,
    nfe=NFE,
    seed=DEFAULT_SEED + 1,
    device=DEVICE,
    max_eval_groups=MAX_EVAL_GROUPS,
)
display(aggregate_metric_table(split_b_metrics))


## 4. Split C held-out compound


In [ ]:
split_c_metrics, split_c_cache = evaluate_sciplex_split(
    states["Split C held-out compound"].X_pca,
    states["Split C held-out compound"].metadata,
    rdkit_by_compound=rdkit_by_compound,
    split_name="Split C held-out compound",
    training_steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE,
    nfe=NFE,
    seed=DEFAULT_SEED + 2,
    device=DEVICE,
    max_eval_groups=MAX_EVAL_GROUPS,
)
display(aggregate_metric_table(split_c_metrics))


## 5. Merge tables and save perturbation figures


In [ ]:
sciplex_metrics = pd.concat([split_b_metrics, split_c_metrics], ignore_index=True)
sciplex_summary = aggregate_metric_table(sciplex_metrics)
save_csv(OUT_DIR / "sciplex_metrics_by_group.csv", sciplex_metrics)
save_csv(OUT_DIR / "sciplex_metrics_summary.csv", sciplex_summary)

fig, ax = plt.subplots(figsize=(10.5, 4.2))
plot_df = sciplex_summary.copy()
plot_df["label"] = plot_df["split_name"].str.replace("Split ", "S", regex=False) + "\n" + plot_df["method"]
ax.bar(plot_df["label"], plot_df["program_readout_sliced_w2"], color="#4C78A8")
ax.set_ylabel("Sliced W2 in split-aware PCA-30")
ax.set_title("sci-Plex perturbation response metrics by split and method")
ax.tick_params(axis="x", rotation=75)
save_figure(fig, "fig_5_3_sciplex_heldout_compound_summary.png")

heldout_keys = [key for key in split_c_cache["predictions"] if key[0] == heldout_compound]
if not heldout_keys:
    heldout_keys = list(split_c_cache["predictions"].keys())
representative_key = sorted(heldout_keys, key=lambda x: x[1])[-1]
panel = split_c_cache["predictions"][representative_key]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharex=True, sharey=True)
panels = [
    ("vehicle", panel["vehicle_as_prediction"], "#8E8E8E"),
    ("ground truth", panel["target"], "#F58518"),
    ("M4 chemistry", panel["M4_chemistry_aware"], "#54A24B"),
    ("M3 no chemistry", panel["M3_no_chemistry"], "#4C78A8"),
]
for ax, (title, pts, color) in zip(axes, panels):
    pts = np.asarray(pts)
    ax.scatter(pts[:, 0], pts[:, 1], s=7, alpha=0.55, linewidths=0, color=color)
    ax.set_title(title)
    ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2")
fig.suptitle(f"Split C held-out {representative_key[0]} dose={representative_key[1]:g}")
save_figure(fig, "fig_5_3_sciplex_heldout_compound.png")

RUN_SUMMARY["sciplex_metrics_summary"] = sciplex_summary
RUN_SUMMARY["sciplex_representative_heldout"] = {"compound": representative_key[0], "dose": representative_key[1]}
display(sciplex_summary)


## 6. Write perturbation run summary and final checks


In [ ]:
required_paths = [
    FIG_DIR / "fig_5_3_sciplex_heldout_compound_summary.png",
    FIG_DIR / "fig_5_3_sciplex_heldout_compound.png",
    TABLE_DIR / "tab_5_2_sciplex_splits.csv",
    OUT_DIR / "rdkit2d_compound_features.npz",
    OUT_DIR / "rdkit2d_diagnostics.json",
    OUT_DIR / "rdkit2d_audit.csv",
    OUT_DIR / "sciplex_metrics_by_group.csv",
    OUT_DIR / "sciplex_metrics_summary.csv",
    OUT_DIR / "real_data_audit.json",
    OUT_DIR / "run_summary_perturbation_sciplex.json",
]

metric_frames = {
    "split_b_metrics": split_b_metrics,
    "split_c_metrics": split_c_metrics,
    "sciplex_metrics": sciplex_metrics,
    "sciplex_summary": sciplex_summary,
}
finite_checks = {}
for name, frame in metric_frames.items():
    numeric = frame.select_dtypes(include=[np.number])
    finite_checks[name] = bool(np.isfinite(numeric.to_numpy()).all()) if numeric.size else True

RUN_SUMMARY["splits_evaluated"] = [
    "Split B held-out highest dose",
    "Split C held-out compound",
]
RUN_SUMMARY["key_metrics"] = {
    "sciplex_summary": sciplex_summary,
    "representative_heldout": RUN_SUMMARY.get("sciplex_representative_heldout", {}),
}
RUN_SUMMARY["finite_metric_checks"] = finite_checks
RUN_SUMMARY["expected_artifacts"] = [str(path.relative_to(PROJECT_ROOT)) for path in required_paths]
if bool(RUN_SUMMARY["sciplex_data"]["summary"].get("is_synthetic", False)):
    raise ValueError("Synthetic sci-Plex data reached final summary; refusing to write perturbation run summary.")
if "synthetic" in str(RUN_SUMMARY["sciplex_data"]["summary"].get("source", "")).lower():
    raise ValueError("Synthetic-labeled sci-Plex source reached final summary; refusing to write perturbation run summary.")
save_json(OUT_DIR / "run_summary_perturbation_sciplex.json", RUN_SUMMARY)

missing = []
for path in required_paths:
    if not path.exists() or path.stat().st_size <= 0:
        missing.append(str(path))
if missing:
    raise FileNotFoundError(f"Missing or empty required artifacts: {missing}")
if not all(finite_checks.values()):
    raise ValueError(f"Non-finite numeric metrics detected: {finite_checks}")

print("Required artifacts:")
for path in required_paths:
    print(path.relative_to(PROJECT_ROOT), path.stat().st_size)
display(pd.DataFrame({"metric_frame": list(finite_checks), "all_finite": list(finite_checks.values())}))
